In [1]:
%run ../../Latex_macros.ipynb

<IPython.core.display.Latex object>

# Appendix to policy gradient II: generalized advantage estimation (GAE)

## Where this fits

- **For your own review**, useful before the PPO session, which uses GAE
- **Position:** Part B, session 2 (the weight slot), appendix
- **What is here**
    - the $p$-step advantage and the TD error it builds on
    - GAE: an average of all $p$-step advantages, set by one number $\lambda$
    - a worked four-step example, and the bias and variance of the $\lambda$ dial
- **You need:** the critic $\statevalfun_\phi$ and the TD error $\delta_\tt$ from the main session
    - the $p$-step idea from the Monte Carlo and TD session


## The TD error and the $p$-step advantage

**Why this section?** GAE is built from two pieces we already have: the TD error, and sampling $p$ rewards before closing with the critic.

- the **TD error** (the *local surprise*): how much better the step went than the critic predicted

$$
\delta_\tt = \rewseq_{\tt+1} + \disc \, \statevalfun_\phi(\stateseq_{\tt+1}) - \statevalfun_\phi(\stateseq_\tt)
$$

- the **$p$-step advantage**: sample $p$ rewards, then use the critic for the rest

$$
\advseq^{(p)}_\tt = \sum_{k=0}^{p-1} \disc^k \, \rewseq_{\tt+k+1} + \disc^p \, \statevalfun_\phi(\stateseq_{\tt+p}) - \statevalfun_\phi(\stateseq_\tt)
$$

- $p = 1$ is $\delta_\tt$
- $p$ up to the end of the episode is the full return minus $\statevalfun_\phi(\stateseq_\tt)$
- **identity** (the value terms telescope): $\advseq^{(p)}_\tt = \sum_{l=0}^{p-1} \disc^l \, \delta_{\tt+l}$


## GAE: all $p$-step advantages, averaged

**Why this section?** Which $p$? Small $p$ has low variance and more bias. Large $p$ the reverse. GAE avoids choosing by averaging all of them.

- weight the $p$-step advantages by $(1-\lambda)\lambda^{p-1}$, for $\lambda \in [0, 1]$, and sum over $p$
- using the identity, this collapses to a sum of TD errors

$$
\hat{\advseq}_\tt = \sum_{l=0}^{\infty} (\disc \lambda)^l \, \delta_{\tt+l}
$$

- (for $\lambda = 1$ the weights $(1-\lambda)\lambda^{p-1}$ vanish, so take the limit: the result is the longest $p$-step advantage, the full return minus the critic)
- the sum stops at the end of the episode (the value of a terminal state is $0$)
- **computed backward** in one pass over an episode: $\hat{\advseq}_\tt = \delta_\tt + \disc \lambda \, \hat{\advseq}_{\tt+1}$, with $\hat{\advseq} = 0$ after the last step
- special cases
    - $\lambda = 0$: $\hat{\advseq}_\tt = \delta_\tt$, the one-step TD advantage
    - $\lambda = 1$: $\hat{\advseq}_\tt = \rlret_\tt - \statevalfun_\phi(\stateseq_\tt)$, the full return minus the critic as a baseline
- source: Schulman et al., 2015


**A worked example** ($\disc = 0.9$, four steps, reward only at the end)

<table>
<thead><tr><th nowrap><div align="left">step $\tt$</div></th><th nowrap><div align="center">0</div></th><th nowrap><div align="center">1</div></th><th nowrap><div align="center">2</div></th><th nowrap><div align="center">3</div></th></tr></thead>
<tbody>
<tr><td nowrap><div align="left">reward $\rewseq_{\tt+1}$</div></td><td nowrap><div align="center">0</div></td><td nowrap><div align="center">0</div></td><td nowrap><div align="center">0</div></td><td nowrap><div align="center">1</div></td></tr>
<tr><td nowrap><div align="left">critic $\statevalfun_\phi(\stateseq_\tt)$</div></td><td nowrap><div align="center">0.5</div></td><td nowrap><div align="center">0.6</div></td><td nowrap><div align="center">0.7</div></td><td nowrap><div align="center">0.8</div></td></tr>
<tr><td nowrap><div align="left">TD error $\delta_\tt$</div></td><td nowrap><div align="center">0.04</div></td><td nowrap><div align="center">0.03</div></td><td nowrap><div align="center">0.02</div></td><td nowrap><div align="center">0.20</div></td></tr>
<tr><td nowrap><div align="left">GAE, $\lambda = 0$</div></td><td nowrap><div align="center">0.04</div></td><td nowrap><div align="center">0.03</div></td><td nowrap><div align="center">0.02</div></td><td nowrap><div align="center">0.20</div></td></tr>
<tr><td nowrap><div align="left">GAE, $\lambda = 0.9$</div></td><td nowrap><div align="center">0.184</div></td><td nowrap><div align="center">0.177</div></td><td nowrap><div align="center">0.182</div></td><td nowrap><div align="center">0.20</div></td></tr>
<tr><td nowrap><div align="left">GAE, $\lambda = 1$ ($= \rlret_\tt - \statevalfun_\phi$)</div></td><td nowrap><div align="center">0.229</div></td><td nowrap><div align="center">0.210</div></td><td nowrap><div align="center">0.20</div></td><td nowrap><div align="center">0.20</div></td></tr>
</tbody></table>

- the critic is *too low at every state* (the true values are $0.729, 0.81, 0.9, 1.0$; its errors are $-0.229, -0.21, -0.2, -0.2$), so every $\delta_\tt$ is positive
- the largest surprise, $0.20$, is at the last step
- note step $0$: with $\lambda = 0$ it gets only $0.04$ although the critic's error there is $0.229$
- the information from the end of the episode reaches it only as $\lambda$ grows
- with $\lambda = 0$ each step is credited only with its own surprise
- with $\lambda = 1$ the later surprises also reach the earlier steps (discounted): the single reward is spread back over the steps
- $\lambda = 0.9$ sits between: the earlier steps get most of the credit, with less of the noise a long sample brings
- (the terminal value is $0$, so $\delta_3 = 1 + 0 - 0.8 = 0.2$; the other $\delta_\tt = 0.9 \, \statevalfun_\phi(\stateseq_{\tt+1}) - \statevalfun_\phi(\stateseq_\tt)$)


## Bias and variance of the $\lambda$ dial

**Why this section?** $\lambda$ is a bias-variance knob, the same trade as the depth dial in sampling.

<table>
<thead><tr><th nowrap><div align="left">$\lambda$</div></th><th nowrap><div align="left">Behaves like</div></th><th nowrap><div align="left">Bias</div></th><th nowrap><div align="left">Variance</div></th></tr></thead>
<tbody>
<tr><td nowrap><div align="left">$0$</div></td><td nowrap><div align="left">one-step TD advantage</div></td><td nowrap><div align="left">from the critic's error</div></td><td nowrap><div align="left">lowest</div></td></tr>
<tr><td nowrap><div align="left">in between</div></td><td nowrap><div align="left">average of $p$-step advantages</div></td><td nowrap><div align="left">partial</div></td><td nowrap><div align="left">partial</div></td></tr>
<tr><td nowrap><div align="left">$1$</div></td><td nowrap><div align="left">full return minus a baseline</div></td><td nowrap><div align="left">none (the baseline depends only on the state)</div></td><td nowrap><div align="left">highest</div></td></tr>
</tbody></table>

- the bias is real only when the critic is wrong
- a better critic makes small $\lambda$ safer
- $\disc < 1$ itself also biases the gradient relative to the undiscounted objective
- in practice $\lambda$ near $0.95$ is common, chosen by experiment
- the discount $\disc$ also shortens the effective horizon: both $\disc$ and $\lambda$ appear together as $\disc \lambda$


# What we established, and what it sets up

- **Established**
    - $\advseq^{(p)}_\tt = \sum_{l<p} \disc^l \delta_{\tt+l}$: the $p$-step advantage is a sum of TD errors
    - GAE $= \sum_l (\disc\lambda)^l \delta_{\tt+l}$: all $p$ averaged
    - $\lambda = 0$ is TD, $\lambda = 1$ is the full return minus a baseline
    - computed in one backward pass
    - $\lambda$ trades bias against variance
- **Sets up**
    - PPO uses GAE as its advantage, together with a critic
